# Solar Filament Segmentation 2026: U-Net baseline

Runs the full pipeline from the GitHub repo: check the data, train, tune post-processing on
validation Panoptic Quality, predict the test set and validate the submission file.

**Before running:** add the competition data (*Add Input > Competitions > Solar Filament
Segmentation Challenge 2026*), turn on a GPU (*Settings > Accelerator > GPU T4 x2 or P100*)
and turn Internet on (*Settings > Internet*) so the repo can be cloned.

In [ ]:
REPO_URL = "https://github.com/FarnooshMemari/solar-filament-segmentation.git"
REPO = "/tmp/solar-filament-segmentation"  # outside /kaggle/working, so it isn't saved as output
RUN = "/kaggle/working/runs/baseline"

import os
if not os.path.exists(REPO):
    !git clone -q {REPO_URL} {REPO}
%cd {REPO}
!pip install -q pycocotools
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Check what the loader finds
Confirms paths, annotation format, annotators per image and image size.

In [ ]:
!python scripts/inspect_data.py

## 2. Train
About 50 minutes on one T4 GPU with the defaults. Watch `val_dice` climb.

In [ ]:
!python scripts/train.py --out {RUN}

## 3. Tune post-processing on validation PQ
Searches thresholds, gap closing, hole filling, fragment merging and minimum area with
8-view test-time augmentation, and saves the best settings to `postprocess.json`.

In [ ]:
!python scripts/tune.py --run {RUN} --tta 8 --workers 4

## 4. Predict the test set and check the file

In [ ]:
!python scripts/predict.py --run {RUN} --out /kaggle/working/submission.csv
!python scripts/check_submission.py --csv /kaggle/working/submission.csv

## 5. Figures for the README and report

In [ ]:
!python scripts/visualize.py --run {RUN} --n 4 --out /kaggle/working/figures
from IPython.display import Image, display
import glob
for path in sorted(glob.glob("/kaggle/working/figures/*.png"))[:2]:
    display(Image(path))

## 6. Submit

Use **Save Version > Save & Run All**, then submit `submission.csv` from the notebook's
Output tab, or download it and upload it on the competition's *Submit Predictions* page.
Only submit files that print `submission=ok` above, since rejected uploads still count
toward the daily limit.

Optional: retrain on every image for the final model with
`python scripts/train.py --out /kaggle/working/runs/full --all-data`, then predict with
`--params {RUN}/postprocess.json` to reuse the tuned settings.